# Differential Attention (V2): Interactive Visual Explorer

> Softmax attention spreads a small amount of probability over every non-matching token. Over 100k tokens that noise adds up and drowns the signal. Differential Transformer (Ye et al., ICLR 2025) fixes it by computing attention as the difference of two softmaxes, subtracting the shared noise floor. DIFF V2 (Microsoft, January 2026) is the production-stack rewrite: matching decode latency to baseline Transformer, no custom kernels, FlashAttention-compatible. This lesson is V1 to V2 end-to-end, with a working toy implementation of the difference operation you can run in stdlib Python.

Welcome to the interactive companion notebook for **Differential Attention (V2)**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Differential attention (Ye et al., ICLR 2025) in stdlib Python.

Builds two softmax maps from split Q, K, subtracts the second from the first
scaled by a learned lambda, multiplies by V. Measures the signal-to-noise
ratio of the resulting attention weights on a synthetic long-context query
and compares to standard softmax attention. Also prints the parameter-count
diff for DIFF V1 and DIFF V2 against a baseline Transformer.


In [ ]:
Pure stdlib. No numpy, no torch.
"""

from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import List

def dot(a: List[float], b: List[float]) -> float:
    return sum(ai * bi for ai, bi in zip(a, b))

def softmax_row(row: List[float]) -> List[float]:
    m = max(row)
    exps = [math.exp(x - m) for x in row]
    s = sum(exps)
    return [e / s for e in exps]


In [ ]:
def standard_attention(Q: List[List[float]], K: List[List[float]],
                       V: List[List[float]]) -> tuple[List[List[float]], List[List[float]]]:
    d = len(Q[0])
    scale = math.sqrt(d)
    weights = []
    for q in Q:
        row = [dot(q, k) / scale for k in K]
        weights.append(softmax_row(row))
    out = []
    d_v = len(V[0])
    for w in weights:
        o = [sum(w[j] * V[j][c] for j in range(len(V))) for c in range(d_v)]
        out.append(o)
    return weights, out


In [ ]:
def diff_attention(Q1: List[List[float]], K1: List[List[float]],
                   Q2: List[List[float]], K2: List[List[float]],
                   V: List[List[float]],
                   lam: float) -> tuple[List[List[float]], List[List[float]]]:
    """Differential attention:
        A1 = softmax(Q1 K1^T / sqrt(d))
        A2 = softmax(Q2 K2^T / sqrt(d))
        out = (A1 - lam * A2) V
    """
    d = len(Q1[0])
    scale = math.sqrt(d)
    weights = []
    for q1, q2 in zip(Q1, Q2):
        row1 = softmax_row([dot(q1, k) / scale for k in K1])
        row2 = softmax_row([dot(q2, k) / scale for k in K2])
        diff = [a - lam * b for a, b in zip(row1, row2)]
        weights.append(diff)
    out = []
    d_v = len(V[0])
    for w in weights:
        o = [sum(w[j] * V[j][c] for j in range(len(V))) for c in range(d_v)]
        out.append(o)
    return weights, out


In [ ]:
def random_projection(d_in: int, d_out: int,
                      rng: random.Random) -> List[List[float]]:
    """d_in x d_out projection matrix with unit-variance columns."""
    return [[rng.gauss(0, 1.0 / math.sqrt(d_in)) for _ in range(d_out)]
            for _ in range(d_in)]

def matmul(X: List[List[float]], W: List[List[float]]) -> List[List[float]]:
    out = []
    d_out = len(W[0])
    for row in X:
        o = [sum(row[k] * W[k][c] for k in range(len(row))) for c in range(d_out)]
        out.append(o)
    return out


In [ ]:
def build_signal_plus_noise(
    n_tokens: int, signal_pos: int, d_embed: int, noise_scale: float,
    rng: random.Random,
) -> tuple[List[List[float]], List[float]]:
    """Return an input embedding sequence X[n_tokens][d_embed] and a query
    vector q. Position signal_pos carries a specific pattern; the query is
    aligned to that pattern. Every other position is Gaussian noise.

The Q, K projections are applied AFTER this build step so that both
    differential branches see the same underlying sequence but project it
    through different matrices — the faithful simulation of DIFF attention.
    """
    pattern = [rng.gauss(0, 1) for _ in range(d_embed)]
    norm = math.sqrt(sum(x * x for x in pattern))
    pattern = [x / norm for x in pattern]
    X = []
    for i in range(n_tokens):
        if i == signal_pos:
            X.append([p + rng.gauss(0, noise_scale * 0.1) for p in pattern])
        else:
            X.append([rng.gauss(0, noise_scale) for _ in range(d_embed)])
    q = list(pattern)
    return X, q


In [ ]:
def snr(weights_row: List[float], signal_pos: int) -> float:
    sig = abs(weights_row[signal_pos])
    noise_vals = [abs(w) for i, w in enumerate(weights_row) if i != signal_pos]
    mean_noise = sum(noise_vals) / len(noise_vals)
    if mean_noise == 0:
        return float("inf")
    return sig / mean_noise

@dataclass
class ParamDiff:
    baseline: int
    diff_v1: int
    diff_v2: int
    extra_v1: int
    extra_v2: int


In [ ]:
def attention_params_baseline(hidden: int) -> int:
    return 4 * hidden * hidden

def attention_params_diff_v1(hidden: int, n_heads: int, d_head: int) -> int:
    q_params = 2 * (hidden * (n_heads * d_head // 2))
    k_params = 2 * (hidden * (n_heads * d_head // 2))
    v_params = hidden * hidden
    o_params = hidden * hidden
    lam_params = 4 * n_heads * (d_head // 2)
    return q_params + k_params + v_params + o_params + lam_params


In [ ]:
def attention_params_diff_v2(hidden: int, n_heads: int, d_head: int,
                             kv_heads: int) -> int:
    q_params = hidden * (2 * n_heads * d_head)
    k_params = hidden * (kv_heads * d_head)
    v_params = hidden * (kv_heads * d_head)
    o_params = (2 * n_heads * d_head) * hidden
    lam_params = 4 * n_heads * d_head
    return q_params + k_params + v_params + o_params + lam_params


In [ ]:
def compute_param_diff(hidden: int, n_heads: int, kv_heads: int) -> ParamDiff:
    d_head = hidden // n_heads
    base = attention_params_baseline(hidden)
    v1 = attention_params_diff_v1(hidden, n_heads, d_head)
    v2 = attention_params_diff_v2(hidden, n_heads, d_head, kv_heads)
    return ParamDiff(
        baseline=base,
        diff_v1=v1,
        diff_v2=v2,
        extra_v1=v1 - base,
        extra_v2=v2 - base,
    )


In [ ]:
def fmt_m(n: int) -> str:
    if n >= 1_000_000:
        return f"{n / 1e6:.1f}M"
    if n >= 1_000:
        return f"{n / 1e3:.1f}K"
    return f"{n}"

def main() -> None:
    rng = random.Random(17)
    print("=" * 70)
    print("DIFFERENTIAL ATTENTION V2 (Phase 10, Lesson 16)")
    print("=" * 70)
    print()

n_tokens = 1024
    signal_pos = 500

print("-" * 70)
    print(f"Step 1: direct-logit toy on length {n_tokens}, signal at pos {signal_pos}")
    print("-" * 70)
    print("  Both branches compute softmax over q.K logits. Branch 1 is a")
    print("  TRAINED head that correctly amplifies the signal. Branch 2 is")
    print("  an untrained/noise-seeing head. DIFF subtracts the shared")
    print("  noise-floor component.")
    print()


In [ ]:
signal_logit = 4.0
    noise_std = 0.5
    logits_trained = [rng.gauss(0, noise_std) for _ in range(n_tokens)]
    logits_trained[signal_pos] = signal_logit
    logits_untrained = [rng.gauss(0, noise_std) for _ in range(n_tokens)]

A1 = softmax_row(logits_trained)
    A2 = softmax_row(logits_untrained)

std_snr = snr(A1, signal_pos)
    std_signal = A1[signal_pos]
    std_noise = sum(abs(w) for i, w in enumerate(A1) if i != signal_pos)
    print(f"  standard softmax attention (branch 1 only):")
    print(f"    weight on signal position  : {std_signal:.6f}")
    print(f"    sum of |noise| weights     : {std_noise:.6f}")
    print(f"    signal-to-noise ratio      : {std_snr:.2f}")
    print()


In [ ]:
for lam in (0.0, 0.3, 0.6, 0.8, 1.0):
        diff = [a1 - lam * a2 for a1, a2 in zip(A1, A2)]
        dsnr = snr(diff, signal_pos)
        d_signal = diff[signal_pos]
        d_noise = sum(abs(w) for i, w in enumerate(diff) if i != signal_pos)
        print(f"  differential attention (lambda={lam:.1f}):")
        print(f"    weight on signal position  : {d_signal:+.6f}")
        print(f"    sum of |noise| weights     : {d_noise:.6f}")
        print(f"    signal-to-noise ratio      : {dsnr:.2f}")
    print()


In [ ]:
print("-" * 70)
    print("Step 2: noise-amplitude sweep (higher = noisier context)")
    print("-" * 70)
    print(f"  {'noise_std':>10}  {'std SNR':>9}  {'diff SNR (lam=0.8)':>20}")
    for noise_scale in (0.25, 0.50, 1.0, 1.5, 2.0):
        lrng = random.Random(int(noise_scale * 100))
        l1 = [lrng.gauss(0, noise_scale) for _ in range(n_tokens)]
        l1[signal_pos] = signal_logit
        l2 = [lrng.gauss(0, noise_scale) for _ in range(n_tokens)]
        A1s = softmax_row(l1)
        A2s = softmax_row(l2)
        diff_s = [a - 0.8 * b for a, b in zip(A1s, A2s)]
        s_snr = snr(A1s, signal_pos)
        d_snr = snr(diff_s, signal_pos)
        print(f"  {noise_scale:>10.2f}  {s_snr:>9.2f}  {d_snr:>20.2f}")
    print()


In [ ]:
print("-" * 70)
    print("Step 3: parameter-count diff, 7B-class config")
    print("-" * 70)
    pd = compute_param_diff(hidden=4096, n_heads=32, kv_heads=8)
    print(f"  baseline attention   : {fmt_m(pd.baseline)}")
    print(f"  DIFF V1 attention    : {fmt_m(pd.diff_v1)}  (delta {fmt_m(pd.extra_v1)})")
    print(f"  DIFF V2 attention    : {fmt_m(pd.diff_v2)}  (delta {fmt_m(pd.extra_v2)})")
    print()


In [ ]:
print("takeaway: DIFF attention reliably improves signal-to-noise in long-context")
    print("          queries. V2 brings the parameter cost down and matches baseline")
    print("          decode speed by doubling Q heads rather than halving head_dim.")

if __name__ == "__main__":
    main()
